# Gradient Descent Variants

Variants of gradient descent to make the network converge faster.

In [50]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Problem: Fit a circle

We are going to train a simple feedforward network to fit a circle.

In [51]:
r = torch.linspace(-1.5, 1.5, 100)
x = []
y = []

for i in r:
    for j in r:
        x.append([i, j])
        y.append(1.0 if i**2 + j**2 < 1 else 0.0)
        
x = torch.tensor(x)
y = torch.tensor(y)

In [52]:
def model():
    return nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
        nn.Sigmoid()
    )

In [53]:
min_loss = 0.1
max_epochs = 10000
learning_rate = 0.03
bce_loss = nn.BCELoss()

## Baseline Gradient Descent

Using batched gradient descent as baseline (not doing stochastic/mini batch to make the comparisons apple to apple). Also setting the seed to a constant value so that all variants start with same initial point in the loss surface. 

This converges to minimum loss (0.1) in 4639 epochs, which would be our baseline.

In [54]:
def gradient_descent(x, y, model):
    for i in range(max_epochs):
        y_pred = model(x).squeeze()
        loss = bce_loss(y_pred, y)
        
        if loss.item() < min_loss:
            print(f"Converged at epoch {i} with loss {loss.item()}")
            return
        
        model.zero_grad()
        loss.backward()
        
        with torch.no_grad():
            for param in model.parameters():
                param -= learning_rate * param.grad
    print(f"Final loss: {loss.item()} > {min_loss} after {max_epochs} epochs")

# Set random seed to make the parameter initialization deterministic
torch.manual_seed(0)
m = model()
gradient_descent(x, y, m)

Converged at epoch 4639 with loss 0.0999981090426445


## Momentum based gradient descent

Idea: If the gradient moves in the same direction we can double down on the direction. We accumulate our update parameter on each iteration and use certain fraction of it along with current gradient.

$u_t = \beta u_{t-1} + \eta \triangledown w$

$w_t = w_{t-1} - u_t$

Converges to min loss in 507 epochs, which is a huge improvement.

In [55]:
def momentum_gradient_descent(x, y, model, beta=0.9):
    previous = [torch.zeros_like(param) for param in model.parameters()]

    for i in range(max_epochs):
        y_pred = model(x).squeeze()
        loss = bce_loss(y_pred, y)
        
        if loss.item() < min_loss:
            print(f"Converged at epoch {i} with loss {loss.item()}")
            return
        
        model.zero_grad()
        loss.backward()
        
        with torch.no_grad():
            for param, prev in zip(model.parameters(), previous):
                update = beta * prev + learning_rate * param.grad
                param -= update
                prev.copy_(update)

    print(f"Final loss: {loss.item()} > {min_loss} after {max_epochs} epochs")

torch.manual_seed(0)
m = model()
momentum_gradient_descent(x, y, m)
            

Converged at epoch 507 with loss 0.09991993755102158


## Nesterov Gradient Descent

Idea: In momentum based gradient descent we are going to update a fixed value no matter what the gradient is, we could as well find the gradient after updating it to the new parameter.

$w_{lookahead} = w_{t-1} - \beta u_{t-1}$

$u_t = \beta u_{t-1} + \eta \triangledown w_{lookahead}$

$w_{t} = w_{t - 1} - u_t$

This converges in 295 iteration almost half of the momentum based.

In [56]:
def nesterov_gradient_descent(x, y, model, beta=0.9):
    previous = [torch.zeros_like(param) for param in model.parameters()]

    for i in range(max_epochs):
        with torch.no_grad():
            for param, prev in zip(model.parameters(), previous):
                prev *= beta
                param -= prev

        with torch.no_grad():
            for param, prev in zip(model.parameters(), previous):
                param -= beta * prev

        y_pred = model(x).squeeze()
        loss = bce_loss(y_pred, y)
        
        if loss.item() < min_loss:
            print(f"Converged at epoch {i} with loss {loss.item()}")
            return
        
        model.zero_grad()
        loss.backward()
        
        with torch.no_grad():
            for param, prev in zip(model.parameters(), previous):
                gradient_update = learning_rate * param.grad
                param -= gradient_update
                prev += gradient_update

    print(f"Final loss: {loss.item()} > {min_loss} after {max_epochs} epochs")

torch.manual_seed(0)
m = model()
nesterov_gradient_descent(x, y, m)

Converged at epoch 295 with loss 0.0997711718082428


## Adagrad

Idea: The learning rate punishes the parameters which gets updates less frequently too much. How about adjusting learning rate on how much a parameter has been updated?

$v_t = v_{t - 1} + (\triangledown w)^2$

$w_{t + 1} = w_t - \frac{\eta}{\sqrt(v_t + \epsilon)} \triangledown w$

Note:
- The entire adjustment on $\eta$ is a element wise operation, one per parameter.
- Adjust using the square of gradient update per parameter (don't confuse it with norm squared $u^Tu$), to make it agnostic of sign.
- $\epsilon$ is a small number to make the denominator not go to zero.
- Empirically works better with square root.


This converges in 1439 iterations, not great but works better in models with lot of parameters.

In [76]:
epsilon = 1e-8
def adagrad(x, y, model):
    changes = [torch.zeros_like(param) for param in model.parameters()]

    for i in range(max_epochs):
        y_pred = model(x).squeeze()
        loss = bce_loss(y_pred, y)
        
        if loss.item() < min_loss:
            print(f"Converged at epoch {i} with loss {loss.item()}")
            return
        
        model.zero_grad()
        loss.backward()
        
        with torch.no_grad():
            for param, rd in zip(model.parameters(), changes):
                rd += param.grad**2
                update = (learning_rate / torch.sqrt(rd + epsilon)) * param.grad
                param -= update

    print(f"Final loss: {loss.item()} > {min_loss} after {max_epochs} epochs")

torch.manual_seed(0)
m = model()
adagrad(x, y, m)

Converged at epoch 1439 with loss 0.09999644011259079


## RMS Prop

Idea: In adagrad, some of the parameters' learning rate get so small they don't get updated at all after some time. It would be better to not reduce it too much, so exponential weighted moving average to the rescue.

$ v_t = \beta v_{t-1} + (1 - \beta) (\triangledown w)^2 $

$ w_{t + 1} = w_t - \frac{eta}{\sqrt(v_t + \epsilon)} \triangledown w$

This converges much faster, 124 iterations

In [74]:
def rms_prop(x, y, model, beta=0.9):
    previous = [torch.zeros_like(param) for param in model.parameters()]

    for i in range(max_epochs):
        y_pred = model(x).squeeze()
        loss = bce_loss(y_pred, y)
        
        if loss.item() < min_loss:
            print(f"Converged at epoch {i} with loss {loss.item()}")
            return
        
        model.zero_grad()
        loss.backward()
        
        with torch.no_grad():
            for param, prev in zip(model.parameters(), previous):
                prev *= beta
                prev += (1 - beta) * param.grad**2
                update = (learning_rate / torch.sqrt(prev + epsilon)) * param.grad
                param -= update
                prev.copy_(prev)

    print(f"Final loss: {loss.item()} > {min_loss} after {max_epochs} epochs")

torch.manual_seed(0)
m = model()
rms_prop(x, y, m)

Converged at epoch 124 with loss 0.09937897324562073


## Adam

Idea: Why not combine RMSProp with Momentum?

$m_t = \beta_1 m_{t - 1} + (1- \beta_1) \triangledown w_{t-1}$

$v_t = \beta_2 v_{t - 1} + (1 - \beta_2) (\triangledown w_{t-1})^2$

$\hat{m_t} = \frac{m_t}{1-\beta_1^t}$

$\hat{v_t} = \frac{v_t}{1-beta_2^t}$

$w_t = w_{t-1} - \frac{\eta}{\sqrt(\hat{v_t} + \epsilon)} \hat{m_t}$


Note:
- $m_t$ - momentum $v_t$ - how much changes
- $\hat{m_t}$ and $\hat{v_t}$ adjusts the respective parameters so that expected update is equal to expected gradient.
- There are some pathological cases where this doesn't converge, but empirically this works very well in most cases and default choice.
- some default choices suggested for $\beta_1 = 0.9$ $\beta_2 = 0.999$ $\epsilon = 1e^{-8}$


This converges in 84 iterations some 50 times faster than naked gradient descent.


In [77]:
def adam(x, y, model, beta1=0.9, beta2=0.999):
    m = [torch.zeros_like(param) for param in model.parameters()]
    v = [torch.zeros_like(param) for param in model.parameters()]

    for i in range(1, max_epochs + 1):
        y_pred = model(x).squeeze()
        loss = bce_loss(y_pred, y)
        
        if loss.item() < min_loss:
            print(f"Converged at epoch {i} with loss {loss.item()}")
            return
        
        model.zero_grad()
        loss.backward()
        
        with torch.no_grad():
            for param, mt, vt in zip(model.parameters(), m, v):
                mt *= beta1
                mt += (1 - beta1) * param.grad
                
                vt *= beta2
                vt += (1 - beta2) * param.grad**2

                m_hat = mt / (1 - beta1**i)
                v_hat = vt / (1 - beta2**i)

                update = (learning_rate / torch.sqrt(v_hat + epsilon)) * m_hat
                param -= update

    print(f"Final loss: {loss.item()} > {min_loss} after {max_epochs} epochs")

torch.manual_seed(0)
m = model()
adam(x, y, m)

Converged at epoch 86 with loss 0.09939073771238327


Lot of these are very intuitive to understand, and most of these optimizers are available in pytorch too.